共通部分

In [ ]:
%load_ext autoreload
%autoreload 2

import datetime
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# プロジェクトルートの設定
PROJECT_ROOT = Path(
    "/Users/hayashikaito/Library/CloudStorage/GoogleDrive-1122hkaito@gmail.com/マイドライブ/jaggle_2026"
)
sys.path.append(str(PROJECT_ROOT))

# モジュールのインポート
from common.catboost.cat_model import run_cat
from common.catboost.cat_model_optuna import run_cat_optuna
from common.lgbm.lgbm_model import run_lgb
from common.lgbm.lgbm_model_optuna import run_lgb_optuna
from common.utils.logger import get_logger
from common.utils.metrics import calculate_logloss  # ★修正: LogLoss をインポート
from common.utils.seed import seed_everything
from common.xgboost.xgb_model import run_xgb
from common.xgboost.xgb_model_optuna import run_xgb_optuna

# 1. 乱数シードの固定
SEED = 42
seed_everything(seed=SEED)

# 2. ターゲット列（10年定着ラベル）とID列の設定
TARGET_COL = "10年定着ラベル"
ID_COL = "社員ID"

# 3. スクリプト名・日付・保存パスの設定
SCRIPT_NAME = "01_baseline"
TODAY = datetime.datetime.now().strftime("%Y%m%d")

LOG_DIR = PROJECT_ROOT / "logs"
logger = get_logger(SCRIPT_NAME, log_dir=str(LOG_DIR))
logger.info(f"=== [{SCRIPT_NAME}] 実験開始 ===")

OUTPUT_DIR = PROJECT_ROOT / "data" / "output" / TODAY
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SUBMISSION_PATH = OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}.csv"

SAVED_MODELS_DIR = PROJECT_ROOT / "saved_models" / TODAY / SCRIPT_NAME

# 4. データの読み込み
INPUT_DIR = PROJECT_ROOT / "data" / "input"

# 4-1. 属性データの読み込み (社員1名 = 1行)
train_persona = pd.read_csv(INPUT_DIR / "employee_persona_train.csv")
test_persona = pd.read_csv(INPUT_DIR / "employee_persona_test.csv")

# 4-2. 月次データの読み込み (社員1名 × 24か月 = 複数行)
train_monthly = pd.read_csv(INPUT_DIR / "employee_monthly_train.csv")
test_monthly = pd.read_csv(INPUT_DIR / "employee_monthly_test.csv")


# 4-3. 月次データを全列横持ちにピボット変換する関数
def transform_monthly_to_wide_all(monthly_df: pd.DataFrame) -> pd.DataFrame:
    """月次データ(0〜23か月)の全カラムを横持ち(ピボット)にし、社員1名=1行に変換する"""
    val_cols = [
        col for col in monthly_df.columns if col not in ["社員ID", "経過月数"]
    ]

    monthly_wide = monthly_df.pivot(
        index="社員ID", columns="経過月数", values=val_cols
    )

    monthly_wide.columns = [
        f"{col}_m{month}" for col, month in monthly_wide.columns
    ]
    monthly_wide = monthly_wide.reset_index()

    return monthly_wide


# 月次データを横持ち化
train_monthly_wide = transform_monthly_to_wide_all(train_monthly)
test_monthly_wide = transform_monthly_to_wide_all(test_monthly)

# 4-4. 属性データと月次横持ちデータを結合 (JOIN)
train_df = pd.merge(train_persona, train_monthly_wide, on="社員ID", how="left")
test_df = pd.merge(test_persona, test_monthly_wide, on="社員ID", how="left")

logger.info(
    f"結合後 Train Shape: {train_df.shape}, Test Shape: {test_df.shape}"
)

基本設定パラメータ（全モデル共通でベースとする設定）

In [ ]:
# ノートブック内の base_params 設定箇所
base_params = {
    "n_splits": 5,
    "seed": SEED,
    "save_dir": str(SAVED_MODELS_DIR),
    "early_stopping_rounds": 20,
    "n_trials": 5,  # ★テスト用: 5回（これなら数秒〜数十秒で終わります）
    "verbose": -1,
}

In [ ]:
# base_params = {
#     "n_splits": 5,
#     "seed": SEED,
#     "save_dir": str(SAVED_MODELS_DIR),
#     "early_stopping_rounds": 50,
#     "n_trials": 10,  # Optunaの試行回数
#     "verbose": -1,
# }

特微量エンジニアリング

In [ ]:
def build_features(
    train: pd.DataFrame, test: pd.DataFrame, target_col: str, id_col: str
):
    """特徴量エンジニアリングを行う関数"""
    train_proc = train.copy()
    test_proc = test.copy()

    # --- 特徴量生成・前処理スペース ---
    # ★ベースライン動作確認のため、モデル入力時にエラーを起こす非数値列（文字列・日付列）を一時除外
    non_num_cols = train_proc.select_dtypes(include=["object"]).columns.tolist()
    
    # 社員ID は後で使うため除外対象から外す
    if id_col in non_num_cols:
        non_num_cols.remove(id_col)
        
    train_proc = train_proc.drop(columns=non_num_cols, errors="ignore")
    test_proc = test_proc.drop(columns=non_num_cols, errors="ignore")
    # ----------------------------------

    # 定義されたターゲット列とID列を使って切り分け
    X_train = train_proc.drop(columns=[target_col, id_col], errors="ignore")
    y_train = train_proc[target_col]
    X_test = test_proc.drop(columns=[id_col], errors="ignore")

    input_data = {"X_train": X_train, "y_train": y_train, "X_test": X_test}

    return input_data, test_proc[id_col]


# 呼び出し
input_data, test_ids = build_features(
    train_df, test_df, target_col=TARGET_COL, id_col=ID_COL
)

optuna_LGBM

In [ ]:
logger.info("--- Optuna LightGBM チューニング開始 ---")
lgb_opt_params = base_params.copy()
lgb_opt_params.update(
    {
        "objective": "binary",            # ★修正: 二元分類
        "metric": "binary_logloss",       # ★修正: Log Loss
    }
)

opt_lgb_res, best_lgb_params = run_lgb_optuna(
    data=input_data, params=lgb_opt_params
)
logger.info(f"LGBM Best Score: {opt_lgb_res['best_score']:.4f}")

LGMB

In [ ]:
logger.info("--- LightGBM 本学習実行 ---")
lgb_res, _ = run_lgb(data=input_data, params=best_lgb_params)

# ★修正: calculate_logloss を使用
lgb_cv = calculate_logloss(input_data["y_train"], lgb_res["oof_preds"])
logger.info(f"LightGBM CV Score: {lgb_cv:.4f}")

optuna_xgb

In [ ]:
logger.info("--- Optuna XGBoost チューニング開始 ---")
xgb_opt_params = base_params.copy()
xgb_opt_params.update(
    {
        "objective": "binary:logistic",   # ★修正: 二元分類（確率出力）
        "eval_metric": "logloss",         # ★修正: Log Loss
    }
)

opt_xgb_res, best_xgb_params = run_xgb_optuna(
    data=input_data, params=xgb_opt_params
)
logger.info(f"XGBoost Best Score: {opt_xgb_res['best_score']:.4f}")

xgb

In [ ]:
logger.info("--- XGBoost 本学習実行 ---")
xgb_res, _ = run_xgb(data=input_data, params=best_xgb_params)

# ★修正: calculate_logloss を使用
xgb_cv = calculate_logloss(input_data["y_train"], xgb_res["oof_preds"])
logger.info(f"XGBoost CV Score: {xgb_cv:.4f}")

optuna_cat

In [ ]:
logger.info("--- Optuna CatBoost チューニング開始 ---")
cat_opt_params = base_params.copy()
cat_opt_params.update(
    {
        "loss_function": "Logloss",
        "eval_metric": "Logloss",
    }
)

opt_cat_res, best_cat_params = run_cat_optuna(
    data=input_data, params=cat_opt_params
)
logger.info(f"CatBoost Best Score: {opt_cat_res['best_score']:.4f}")

cat

In [ ]:
logger.info("--- CatBoost 本学習実行 ---")
cat_res, _ = run_cat(data=input_data, params=best_cat_params)

# ★修正: calculate_logloss を使用
cat_cv = calculate_logloss(input_data["y_train"], cat_res["oof_preds"])
logger.info(f"CatBoost CV Score: {cat_cv:.4f}")

アンサンブル

In [ ]:
logger.info("--- 3モデルのアンサンブル（単純平均） ---")

oof_ensemble = (
    lgb_res["oof_preds"] + xgb_res["oof_preds"] + cat_res["oof_preds"]
) / 3.0

test_ensemble = (
    lgb_res["test_preds"] + xgb_res["test_preds"] + cat_res["test_preds"]
) / 3.0

# ★修正: calculate_logloss を使用
ensemble_cv = calculate_logloss(input_data["y_train"], oof_ensemble)
logger.info(f"Ensemble CV Score: {ensemble_cv:.4f}")

提出

In [ ]:
sub = pd.DataFrame({ID_COL: test_ids, TARGET_COL: test_ensemble})

sub.to_csv(SUBMISSION_PATH, index=False)
logger.info(f"提出ファイルを保存しました: {SUBMISSION_PATH}")
logger.info("=== 実験完了 ===")